# Notebook 04: Multi-Qubit Registers and Quantum Entanglement

This notebook covers multi-qubit systems, the Controlled-NOT (CNOT) gate, and the construction and verification of the four canonical Bell states.

---

## Learning Objectives
1. Define multi-qubit quantum registers.
2. Apply the Controlled-NOT (CNOT / CX) operation.
3. Construct all four maximally entangled Bell states.
4. Verify non-separability using statevectors and correlation statistics.


---
## Real-World Applications & Modern Use Cases

Entangled quantum states are critical for:
- **Quantum Key Distribution (QKD):** Establishing tamper-evident encryption keys over optical fibers using entanglement-based protocols like E91.
- **Distributed Quantum Computing:** Connecting multiple discrete quantum processor chips via entangled photon links to scale beyond single-chip thermal budgets.


---
## Section 1: Multi-Qubit Computational Basis

For an $n$-qubit system, the Hilbert space has dimension $2^n$. For 2 qubits, the computational basis states are $|00\rangle, |01\rangle, |10\rangle, |11\rangle$.


In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Initialize a 2-qubit register
qc_2q = QuantumCircuit(2)
sv_init = Statevector.from_instruction(qc_2q)

print("2-Qubit Initial Ground State |00>:")
print(sv_init.data)
print("Initial Basis Probabilities:", sv_init.probabilities_dict())


2-Qubit Initial Ground State |00>:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
Initial Basis Probabilities: {'00': 1.0}


---
## Section 2: The Controlled-NOT Gate

The CNOT gate acts on a control qubit and a target qubit. If the control qubit is in state $|1\rangle$, an X gate is applied to the target qubit.


In [2]:
cnot_qc = QuantumCircuit(2)

# Set control qubit (qubit 0) to |1>
cnot_qc.x(0)

# Apply CNOT: control=0, target=1
cnot_qc.cx(0, 1)

print("CNOT Demonstration Circuit:")
print(cnot_qc.draw(output='text'))

sv_cnot = Statevector.from_instruction(cnot_qc)
print("\nResulting state (Qiskit orders qubits |q1, q0>):")
print(sv_cnot.probabilities_dict())


CNOT Demonstration Circuit:
     ┌───┐     
q_0: ┤ X ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

Resulting state (Qiskit orders qubits |q1, q0>):
{'11': 1.0}


---
## Section 3: Constructing Bell State $|\Phi^+\rangle$

The state $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$ is prepared by combining a Hadamard gate with a CNOT gate.


In [3]:
phi_plus = QuantumCircuit(2)
phi_plus.h(0)
phi_plus.cx(0, 1)

print("Bell State |Phi+> Circuit:")
print(phi_plus.draw(output='text'))

sv_phi_plus = Statevector.from_instruction(phi_plus)
print("\nExact Probabilities for |Phi+>:")
for state, prob in sv_phi_plus.probabilities_dict().items():
    print(f"State |{state}>: {prob:.4f}")


Bell State |Phi+> Circuit:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

Exact Probabilities for |Phi+>:
State |00>: 0.5000
State |11>: 0.5000


---
## Section 4: Constructing Bell State $|\Phi^-\rangle$

The state $|\Phi^-\rangle = \frac{|00\rangle - |11\rangle}{\sqrt{2}}$ introduces a relative phase of $\pi$.


In [4]:
phi_minus = QuantumCircuit(2)
phi_minus.x(0)
phi_minus.h(0)
phi_minus.cx(0, 1)

sv_phi_minus = Statevector.from_instruction(phi_minus)
print("Bell State |Phi-> Statevector:")
print(sv_phi_minus.data)
print("Probabilities:", sv_phi_minus.probabilities_dict())


Bell State |Phi-> Statevector:
[ 0.70710678+0.j  0.        +0.j  0.        +0.j -0.70710678+0.j]
Probabilities: {'00': 0.4999999999999999, '11': 0.4999999999999999}


---
## Section 5: Constructing Bell States $|\Psi^+\rangle$ and $|\Psi^-\rangle$

The states $|\Psi^+\rangle = \frac{|01\rangle + |10\rangle}{\sqrt{2}}$ and $|\Psi^-\rangle = \frac{|01\rangle - |10\rangle}{\sqrt{2}}$ represent anti-correlated entangled pairs.


In [5]:
# Construct |Psi+>
psi_plus = QuantumCircuit(2)
psi_plus.x(1)
psi_plus.h(0)
psi_plus.cx(0, 1)

# Construct |Psi->
psi_minus = QuantumCircuit(2)
psi_minus.x(1)
psi_minus.z(0)
psi_minus.h(0)
psi_minus.cx(0, 1)

sv_psi_plus = Statevector.from_instruction(psi_plus)
sv_psi_minus = Statevector.from_instruction(psi_minus)

print("Probabilities for |Psi+>:", sv_psi_plus.probabilities_dict())
print("Probabilities for |Psi->:", sv_psi_minus.probabilities_dict())


Probabilities for |Psi+>: {'01': 0.4999999999999999, '10': 0.4999999999999999}
Probabilities for |Psi->: {'01': 0.4999999999999999, '10': 0.4999999999999999}


---
## Section 6: Verifying Entanglement Non-Separability

A quantum state is entangled if and only if it cannot be factored into the tensor product of two independent single-qubit states:
$$|\psi_{AB}\rangle \ne |\psi_A\rangle \otimes |\psi_B\rangle$$


In [6]:
# Check purity and density matrix
from qiskit.quantum_info import DensityMatrix, entropy

rho_bell = DensityMatrix(sv_phi_plus)
# Partial trace over qubit 1 yields maximally mixed state on qubit 0
from qiskit.quantum_info import partial_trace
rho_reduced = partial_trace(rho_bell, [1])

print("Reduced Density Matrix of Qubit 0:")
print(rho_reduced.data)
print(f"Von Neumann Entropy (1.0 = maximally entangled): {entropy(rho_reduced):.4f}")


Reduced Density Matrix of Qubit 0:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Von Neumann Entropy (1.0 = maximally entangled): 1.0000
